# FP-growth: lab notebook

**Machine Learning (5CS4-05), Unsupervised learning**

Run the cells from top to bottom (Runtime > Run all). NumPy and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
from collections import defaultdict

T = [["Bread", "Milk"], ["Bread", "Diaper", "Beer", "Eggs"], ["Milk", "Diaper", "Beer", "Cola"],
     ["Bread", "Milk", "Diaper", "Beer"], ["Bread", "Milk", "Diaper", "Cola"]]
min_count = 3
ITEMS = ["Bread", "Milk", "Diaper", "Beer", "Eggs", "Cola"]   # tie-break order

class Node:
    def __init__(self, item, parent):
        self.item, self.parent, self.count, self.children = item, parent, 0, {}

### 1. Scan 1: count items and fix the header order

In [ ]:
# 1. Scan 1: count items and fix the header order
cnt = defaultdict(int)
for t in T:
    for i in t:
        cnt[i] += 1
order = sorted([i for i in cnt if cnt[i] >= min_count], key=lambda i: (-cnt[i], ITEMS.index(i)))
print("header order:", [(i, cnt[i]) for i in order])

### 2. Scan 2: build the FP-tree

In [ ]:
# 2. Scan 2: build the FP-tree
def build(transactions):
    root, header = Node(None, None), defaultdict(list)
    for t, w in transactions:
        node = root
        for i in [x for x in order if x in t]:
            if i not in node.children:
                node.children[i] = Node(i, node)
                header[i].append(node.children[i])
            node = node.children[i]
            node.count += w
    return root, header

root, header = build([(set(t), 1) for t in T])
def show(node, depth=0):
    for c in node.children.values():
        print("  " * depth + f"{c.item}:{c.count}")
        show(c, depth + 1)
show(root)

### 3. Mine: conditional pattern bases, from the least frequent item upward

In [ ]:
# 3. Mine: conditional pattern bases, from the least frequent item upward
def mine(header, suffix, out):
    for item in sorted(header, key=lambda i: -order.index(i)):
        total = sum(n.count for n in header[item])
        if total < min_count:
            continue
        pattern = [item] + suffix
        out[tuple(sorted(pattern))] = total
        cpb = []
        for n in header[item]:
            path, p = [], n.parent
            while p.item is not None:
                path.append(p.item); p = p.parent
            if path:
                cpb.append((set(path), n.count))
        if cpb:
            sub_root, sub_header = build(cpb)
            mine(sub_header, pattern, out)

found = {}
mine(header, [], found)
for s, n in sorted(found.items(), key=lambda kv: (len(kv[0]), kv[0])):
    print(s, n)
print("number of frequent itemsets:", len(found))

## Expected output

Running all cells above should print:

```
header order: [('Bread', 4), ('Milk', 4), ('Diaper', 4), ('Beer', 3)]
Bread:4
  Milk:3
    Diaper:2
      Beer:1
  Diaper:1
    Beer:1
Milk:1
  Diaper:1
    Beer:1
('Beer',) 3
('Bread',) 4
('Diaper',) 4
('Milk',) 4
('Beer', 'Diaper') 3
('Bread', 'Diaper') 3
('Bread', 'Milk') 3
('Diaper', 'Milk') 3
number of frequent itemsets: 8
```

## Practice exercises

Try these in the cells below.

1. Build the FP-tree by hand for the first four transactions and compare with the simulator.
2. Raise the minimum support count by one. How many nodes does the tree lose?
3. Run both Apriori and FP-growth in the script and confirm they find the same sets.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
